## Preliminaries

In [0]:
%pip install databricks-langchain
%pip install pyvis

In [0]:
import json
from databricks_langchain.chat_models import ChatDatabricks
from IPython.display import display, Markdown
import mlflow
import warnings

In [0]:
mlflow.tracing.disable()
mlflow.autolog(disable=True)
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic.main")

## Initial Call to Model

In [0]:
chat_model = ChatDatabricks(
    endpoint="databricks-claude-sonnet-4", # could use a variety of models in place of this one
    temperature=0.1, # controls the randomness of the generated text
    max_tokens=1500) # controls length of the response

In [0]:
prompt = "Write a song in the style of Celine Dion about Canada's attitude towards Donald Trump"

In [0]:
response = chat_model.invoke(prompt).content
print(response)

## Extracting Structured Representations of Unstructured Data

We begin by seeing how an LLM can reprsent the content of news articles.  To begin, we use https://www.newyorker.com/news/the-lede/donald-trump-wins-a-second-term whose raw text content is stored on a shared volume.

In [0]:
read_file = "/Volumes/daz_aitraining_cat/aitraining/aitraining_volume/example_text.txt"

with open(read_file, 'r', encoding="utf-8") as file:
    ny_article = file.read()

print(ny_article[:1000])

Extracting information from this raw text requires forming a prompt to the LLM.  The study of how to effectively do so is called prompt engineering.  For example, Anthropic's guide is https://docs.claude.com/en/docs/build-with-claude/prompt-engineering/overview.

### "Be clear and direct"

In [0]:
# PRINCIPLE 1: Be clear and detailed (BAD PROMPT)
prompt1 = f"""
whom does this article talk about?:

{ny_article}
"""

print(prompt1[:100])

In [0]:
chat_model = ChatDatabricks(
    endpoint="databricks-claude-sonnet-4",
    temperature=0.0, # for maximum replicability
    max_tokens=5000)

In [0]:

response = chat_model.invoke(prompt1).content
print(response)

In [0]:
prompt2 = f"""
We want to extract the relevant people from a news article.

Please follow these steps:
1. Identify all the people mentioned and any description of them
2. Identify any political offices mentioned

Here is the text of the article:
{ny_article}

"""

In [0]:
response = chat_model.invoke(prompt2).content
print(response)

Modern LLMs also support prompts that directly incorporate data schema which can help clarify and organize what information you want.  Some even have an explicit function mode that guarantees a particular output.

In [0]:
prompt3 = f"""
We want to extract the relevant characters from a news article. 

Please provide your output in the following JSON format:

{{
  "people": [
    {{
      "name": "person's full name",
      "description": "their role or description from the article"
    }}
  ],
  "institutions": [
    {{
      "name": "institution name",
      "type": "type of institution (e.g., government, media, party, etc.)",
      "context": "brief context of how it's mentioned"
    }}
  ]
}}

Here is the text of the article:
{ny_article}
"""

In [0]:
response = chat_model.invoke(prompt3).content
print(response)

These data schema can fundamentally change how input text is represented: temporal, network, etc.  Below we illustrate a network-structured prompt and associated visualization.

In [0]:
# different prompts can generate different representations of the data

prompt4 = f"""
Create a network graph representation of the article.

Return JSON:

{{
  "nodes": [
    {{"id": "trump", "label": "Donald Trump", "type": "person"}},
    {{"id": "gop", "label": "Republican Party", "type": "institution"}}
  ],
  "edges": [
    {{"from": "trump", "to": "gop", "relationship": "leads"}},
    {{"from": "trump", "to": "harris", "relationship": "defeated"}}
  ]
}}

Article: {ny_article}
"""

In [0]:
response = chat_model.invoke(prompt4).content
print(response)

In [0]:
import re
from pyvis.network import Network

# Get and clean response
response = chat_model.invoke(prompt4).content
json_str = re.sub(r'^```json\n|```$', '', response.strip(), flags=re.MULTILINE)
data = json.loads(json_str)

# Color mapping
color_map = {
    'person': '#3498db',
    'institution': '#2ecc71',
    'location': '#f39c12',
    'event': '#e74c3c',
    'policy': '#9b59b6'
}

# Create network
net = Network(
    height='800px', 
    width='100%', 
    directed=True, 
    notebook=True,
    cdn_resources='in_line'
)

# Add nodes
for node in data['nodes']:
    net.add_node(node['id'], label=node['label'], 
                 color=color_map.get(node['type'], '#95a5a6'),
                 title=f"{node['label']} ({node['type']})")

# Add edges
for edge in data['edges']:
    net.add_edge(edge['from'], edge['to'], 
                 title=edge['relationship'], arrows='to')

# Configure and save
net.toggle_physics(True)
net.show('network.html')

print("✓ Network HTML generated!")

In [0]:
# Display in Databricks
with open('network.html', 'r') as f:
    html_content = f.read()
displayHTML(html_content)

### System Prompts 

A system prompt allows you to endow the LLM with a "persona" which guides what output is generated.  We'll illustrate this with a Texas Tribune article about a Venezuelan gang https://www.texastribune.org/2024/09/18/texas-venezuelan-gang-tren-de-aragua-abbott-crackdown/.

In [0]:
read_file = "/Volumes/daz_aitraining_cat/aitraining/aitraining_volume/example_text2.txt"

with open(read_file, 'r', encoding="utf-8") as file:
    tt_article = file.read()

my_prompt = f"""Summarize the article below.
Article:
{tt_article}
"""

In [0]:
from langchain_core.messages import SystemMessage, HumanMessage

chat_model = ChatDatabricks(
    endpoint="databricks-claude-sonnet-4",  # or whatever endpoint you're using
    temperature=0.0,
    max_tokens=100
)

response = chat_model.invoke([
    SystemMessage(content="You are a helpful assistant that replies with a concise one-sentence answer that always starts with the letter T."),
    HumanMessage(content=my_prompt)
])

# Access the content
print(response.content)

In [0]:
response = chat_model.invoke([
    SystemMessage(content="You are a language model that works with young children. Never produce content related to violence or gangs.  If asked to produce this content please reply with the phrase \"I can\'t do that :( \nViolence is not good\""),
    HumanMessage(content=my_prompt)
])

# Access the content
print(response.content)